# Award classifications, phase 1: IDRC's own sector codes

Builds `openalex.awards.award_classifications_raw`, the side table `nightly_grants/lib/award_research_scope.py` reads (oxjob #1570,
funding-flows work under the Wellcome funding-metadata grant 323416/Z/24/Z): one row per award and classification code, keyed by the
producer's STAGING triple. Scheme `oecd_dac_purpose` (IATI sector vocabulary 1). An award carrying a research purpose code
(`research/dac_research_codes.csv`: 11182, 12182, 43082 ...) is research funding whatever its recipient.
Phase 1 carries IDRC's codes from `idrc_raw.sectors_json` (5,112 activities, 1,649 with a research code on 2026-10-09). Rerun whenever
`idrc_raw` is refreshed (the table is rebuilt whole). The nightly stops if a provenance's rows or research-coded rows fall by more than 1%.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.award_classifications_raw AS
WITH idrc AS (SELECT iati_identifier, from_json(sectors_json, 'array<struct<code:string,vocabulary:string,percentage:string>>') ss FROM openalex.awards.idrc_raw)
SELECT DISTINCT 'idrc_iati' provenance, iati_identifier funder_award_id,
       abs(xxhash64(concat('4320319949', ':', lower(iati_identifier)))) % 9000000000 staging_id, 4320319949L funder_id, lower(iati_identifier) award_key,
       'oecd_dac_purpose' scheme, trim(s.code) code, 'sector' source_field, 'funder' asserted_by, 'af1570-idrc-sectors-v1' adapter_version, current_timestamp() asserted_at
FROM idrc LATERAL VIEW explode(ss) e AS s WHERE coalesce(s.vocabulary, '1') = '1' AND s.code IS NOT NULL AND trim(s.code) <> '';

In [ ]:
%sql
SELECT provenance, count(*) rows, count(DISTINCT staging_id) awards,
       count_if(scheme='oecd_dac_purpose' AND code IN ('11182','12182','12382','23182','23082','31182','31282','31382','32182','41082','43082')) research_rows
FROM openalex.awards.award_classifications_raw GROUP BY ALL ORDER BY ALL

In [ ]:
%sql
-- Shape the nightly checks (CLASSIFICATION_RAW_SHAPE / CLASSIFICATION_RAW_UNIQUE): nothing may come back.
SELECT * FROM openalex.awards.award_classifications_raw
WHERE NOT coalesce(length(provenance)>0 AND length(funder_award_id)>0 AND staging_id IS NOT NULL AND funder_id IS NOT NULL AND length(award_key)>0
  AND length(scheme)>0 AND length(code)>0 AND asserted_by IN ('funder','openalex','curated') AND length(adapter_version)>0 AND asserted_at IS NOT NULL, false)
UNION ALL
SELECT * EXCEPT(dup) FROM (SELECT *, count(*) OVER (PARTITION BY provenance,staging_id,funder_id,award_key,scheme,code) dup
  FROM openalex.awards.award_classifications_raw) WHERE dup > 1